In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week6_Spark_Assignment") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark Version:", spark.version)
print("SparkSession created successfully!")

Defaulting to user installation because normal site-packages is not writeable
Spark Version: 4.1.2
SparkSession created successfully!


In [2]:
# Q3 - Read CSV with proper schema handling
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv("Sample_-_Superstore.csv")

print(f"Shape: {df.count()} rows x {len(df.columns)} columns")
df.printSchema()
df.show(5)

Shape: 9994 rows x 21 columns
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+--

In [3]:
# Q5 - Select Product ID and Sales where Category = Technology
df_tech = df.select("Product ID", "Sales") \
            .filter(col("Category") == "Technology")

print("Q5 - Technology category filtered:")
df_tech.show(10)
print(f"Total Technology records: {df_tech.count()}")

Q5 - Technology category filtered:
+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
|TEC-PH-10000486| 371.168|
|TEC-PH-10004093| 147.168|
|TEC-AC-10000171|   45.98|
|TEC-AC-10002167|    45.0|
|TEC-PH-10003988|    21.8|
+---------------+--------+
only showing top 10 rows
Total Technology records: 1847


In [4]:
# Q6 - Rename column and cast Sales to Double
df_renamed = df \
    .withColumnRenamed("Product Name", "product_description") \
    .withColumn("Sales", col("Sales").cast(DoubleType()))

print("Q6 - Column renamed and Sales cast to Double:")
df_renamed.select("product_description", "Sales").printSchema()
df_renamed.select("product_description", "Sales").show(5)

Q6 - Column renamed and Sales cast to Double:
root
 |-- product_description: string (nullable = true)
 |-- Sales: double (nullable = true)

+--------------------+--------+
| product_description|   Sales|
+--------------------+--------+
|Bush Somerset Col...|  261.96|
|Hon Deluxe Fabric...|  731.94|
|Self-Adhesive Add...|   14.62|
|Bretford CR4500 S...|957.5775|
|Eldon Fold 'N Rol...|  22.368|
+--------------------+--------+
only showing top 5 rows


In [5]:
# Q8 - Filter Segment=Corporate AND Sales > 1000
df_filtered = df.filter(
    (col("Segment") == "Corporate") &
    (col("Sales") > 1000)
)
print("Q8 - Segment=Corporate AND Sales > 1000:")
df_filtered.select("Order ID", "Segment", "Sales", "Category").show(10)
print(f"Total matching records: {df_filtered.count()}")

Q8 - Segment=Corporate AND Sales > 1000:
+--------------+---------+--------+---------------+
|      Order ID|  Segment|   Sales|       Category|
+--------------+---------+--------+---------------+
|CA-2016-117590|Corporate|1097.544|     Technology|
|CA-2016-105816|Corporate| 1029.95|     Technology|
|CA-2014-106376|Corporate|1113.024|Office Supplies|
|CA-2016-114489|Corporate| 1951.84|      Furniture|
|CA-2015-146262|Corporate|  1188.0|     Technology|
|US-2014-106992|Corporate|3059.982|     Technology|
|US-2014-106992|Corporate|2519.958|     Technology|
|CA-2016-142545|Corporate| 1082.48|Office Supplies|
|CA-2016-155516|Corporate| 1043.92|      Furniture|
|US-2017-134481|Corporate|1488.424|      Furniture|
+--------------+---------+--------+---------------+
only showing top 10 rows
Total matching records: 154


In [6]:
# Q10 - Add new column final_price = Sales * 1.18 (tax)
df_with_tax = df.withColumn(
    "final_price",
    round(col("Sales") * 1.18, 2)
)
print("Q10 - New column final_price (Sales * 1.18 tax):")
df_with_tax.select("Order ID", "Product Name", "Sales", "final_price").show(10)

Q10 - New column final_price (Sales * 1.18 tax):
+--------------+--------------------+--------+-----------+
|      Order ID|        Product Name|   Sales|final_price|
+--------------+--------------------+--------+-----------+
|CA-2016-152156|Bush Somerset Col...|  261.96|     309.11|
|CA-2016-152156|Hon Deluxe Fabric...|  731.94|     863.69|
|CA-2016-138688|Self-Adhesive Add...|   14.62|      17.25|
|US-2015-108966|Bretford CR4500 S...|957.5775|    1129.94|
|US-2015-108966|Eldon Fold 'N Rol...|  22.368|      26.39|
|CA-2014-115812|Eldon Expressions...|   48.86|      57.65|
|CA-2014-115812|          Newell 322|    7.28|       8.59|
|CA-2014-115812|Mitel 5320 IP Pho...| 907.152|    1070.44|
|CA-2014-115812|DXL Angle-View Bi...|  18.504|      21.83|
|CA-2014-115812|Belkin F5C206VTEL...|   114.9|     135.58|
+--------------+--------------------+--------+-----------+
only showing top 10 rows


In [7]:
# Q12 - Pipeline: Read → Filter Nulls → Save
import pandas as pd

print(f"Step 1 - Original rows: {df.count()}")

df_clean = df.filter(col("Customer ID").isNotNull())
print(f"Step 2 - After null filter: {df_clean.count()} rows")

df_clean.toPandas().to_csv("C:/Users/hello/superstore_output.csv", index=False)
print("Step 3 - Saved as CSV!")
print("Step 4 - Pipeline complete!")

df_clean.select("Customer ID", "Customer Name", "Region", "Sales").show(5)

Step 1 - Original rows: 9994
Step 2 - After null filter: 9994 rows
Step 3 - Saved as CSV!
Step 4 - Pipeline complete!
+-----------+---------------+------+--------+
|Customer ID|  Customer Name|Region|   Sales|
+-----------+---------------+------+--------+
|   CG-12520|    Claire Gute| South|  261.96|
|   CG-12520|    Claire Gute| South|  731.94|
|   DV-13045|Darrin Van Huff|  West|   14.62|
|   SO-20335| Sean O'Donnell| South|957.5775|
|   SO-20335| Sean O'Donnell| South|  22.368|
+-----------+---------------+------+--------+
only showing top 5 rows


In [8]:
# Q14 - Filter Region=West OR Segment=Corporate
df_or_filter = df.filter(
    (col("Region") == "West") |
    (col("Segment") == "Corporate")
)
print("Q14 - Region='West' OR Segment='Corporate':")
df_or_filter.select("Order ID", "Region", "Segment", "Sales").show(10)
print(f"Total matching records: {df_or_filter.count()}")

Q14 - Region='West' OR Segment='Corporate':
+--------------+------+---------+--------+
|      Order ID|Region|  Segment|   Sales|
+--------------+------+---------+--------+
|CA-2016-138688|  West|Corporate|   14.62|
|CA-2014-115812|  West| Consumer|   48.86|
|CA-2014-115812|  West| Consumer|    7.28|
|CA-2014-115812|  West| Consumer| 907.152|
|CA-2014-115812|  West| Consumer|  18.504|
|CA-2014-115812|  West| Consumer|   114.9|
|CA-2014-115812|  West| Consumer|1706.184|
|CA-2014-115812|  West| Consumer| 911.424|
|CA-2016-161389|  West| Consumer| 407.976|
|CA-2014-167164|  West| Consumer|    55.5|
+--------------+------+---------+--------+
only showing top 10 rows
Total matching records: 5263


In [9]:
# Q15 - Final pipeline: filter → partition by Region → save
import os

print(f"Raw rows: {df.count()}")

df_valid = df.filter(col("Sales") >= 0)
print(f"After removing negative sales: {df_valid.count()} rows")

os.makedirs("C:/Users/hello/superstore_partitioned", exist_ok=True)
for region in df_valid.select("Region").distinct().collect():
    r = region["Region"]
    folder = f"C:/Users/hello/superstore_partitioned/Region={r}"
    os.makedirs(folder, exist_ok=True)
    df_valid.filter(col("Region") == r).toPandas().to_csv(f"{folder}/part.csv", index=False)
    print(f"Saved: Region={r}")

print("Pipeline complete!")

Raw rows: 9994
After removing negative sales: 9994 rows
Saved: Region=South
Saved: Region=Central
Saved: Region=East
Saved: Region=West
Pipeline complete!


In [10]:
# Null check across all columns
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
null_counts.show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [11]:
# Schema Fix + Derived Column Pipeline
df_cleaned = df \
    .withColumn("Order Date", to_date(col("Order Date"), "M/d/yyyy")) \
    .withColumn("Ship Date", to_date(col("Ship Date"), "M/d/yyyy")) \
    .withColumn("Sales", col("Sales").cast("double")) \
    .withColumn("Profit", col("Profit").cast("double")) \
    .withColumn("Discount", col("Discount").cast("double")) \
    .withColumnRenamed("Sub-Category", "Sub_Category") \
    .withColumnRenamed("Product Name", "Product_Name") \
    .withColumnRenamed("Customer Name", "Customer_Name") \
    .withColumn("Revenue_After_Discount",
        round(col("Sales") * (1 - col("Discount")), 2))

print("Schema after transformation:")
df_cleaned.printSchema()
df_cleaned.select("Order Date", "Sales", "Discount", "Revenue_After_Discount").show(5)

Schema after transformation:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Revenue_After_Discount: double (nullable = true)

+----------+--------+--------+----------------------+
|Order Date

In [12]:
# Final aggregation pipeline → results.csv
df_final = df_cleaned.filter(
    (col("Sales") > 100) & (col("Profit") > 0)
)
print(f"Filtered rows: {df_final.count()}")

df_summary = df_final \
    .groupBy("Region", "Category") \
    .agg(
        round(sum("Sales"), 2).alias("Total_Sales"),
        round(avg("Profit"), 2).alias("Avg_Profit"),
        count("*").alias("Order_Count")
    ).orderBy("Region", "Total_Sales", ascending=False)

df_summary.show()
df_summary.toPandas().to_csv("C:/Users/hello/results.csv", index=False)
print("Saved results.csv!")

Filtered rows: 2896
+-------+---------------+-----------+----------+-----------+
| Region|       Category|Total_Sales|Avg_Profit|Order_Count|
+-------+---------------+-----------+----------+-----------+
|   West|     Technology|  233270.46|    128.16|        380|
|   West|      Furniture|  182790.86|     68.83|        314|
|   West|Office Supplies|  166357.65|    110.42|        379|
|  South|     Technology|   103195.2|    164.99|        162|
|  South|Office Supplies|   85348.83|    124.49|        191|
|  South|      Furniture|   78099.34|    107.63|        145|
|   East|     Technology|  210891.87|    267.67|        248|
|   East|Office Supplies|  141079.28|    129.17|        305|
|   East|      Furniture|  107822.25|     97.06|        202|
|Central|     Technology|  141810.04|    142.23|        251|
|Central|Office Supplies|  109756.34|    162.71|        214|
|Central|      Furniture|    73434.9|    152.86|        105|
+-------+---------------+-----------+----------+-----------+

Sav

In [13]:
# Final Pipeline Output
df_summary.show()
print("Saved results.csv successfully!")

+-------+---------------+-----------+----------+-----------+
| Region|       Category|Total_Sales|Avg_Profit|Order_Count|
+-------+---------------+-----------+----------+-----------+
|   West|     Technology|  233270.46|    128.16|        380|
|   West|      Furniture|  182790.86|     68.83|        314|
|   West|Office Supplies|  166357.65|    110.42|        379|
|  South|     Technology|   103195.2|    164.99|        162|
|  South|Office Supplies|   85348.83|    124.49|        191|
|  South|      Furniture|   78099.34|    107.63|        145|
|   East|     Technology|  210891.87|    267.67|        248|
|   East|Office Supplies|  141079.28|    129.17|        305|
|   East|      Furniture|  107822.25|     97.06|        202|
|Central|     Technology|  141810.04|    142.23|        251|
|Central|Office Supplies|  109756.34|    162.71|        214|
|Central|      Furniture|    73434.9|    152.86|        105|
+-------+---------------+-----------+----------+-----------+

Saved results.csv succe